(ch:poo)=
# P00 : programmation orientée objet 

Mis à jour : {sub-ref}`today`, lecture : {sub-ref}`wordcount-minutes`
minutes minimum, PhL.

Ce chapitre complète le cours d'*Algo-Prog* de L1 en introduisant la **programmation orientée objet (POO)** en python.
Il s'appuie sur les notions déjà vues, en particulier :

- les **fonctions** (définition, paramètres typés, valeur de retour) —- voir ce [chapitre](ch:fonctions) 
- les **types composés** (`list`, `dict`, `tuple`, `str`) — voir ce [chapitre](ch:types-omposes) , et en particulier la notion d'**enregistrement** (ou *structure*), qui regroupe des champs nommés (*attributs*) accessibles en notation pointée `objet.attribut`,
- l'**affectation et les références** pour les objets mutables — voir ce [chapitre](ch:fonctions avancees).

**Idée clé.** La POO généralise la notion d'enregistrement : on ne regroupe plus seulement des **données** (des attributs), mais aussi les **traitements** (des fonctions, appelées ici *méthodes*) qui s'appliquent naturellement à ces données, au sein d'une même entité appelée **classe**.

## Pourquoi la programmation orientée objet ?

Reprenons l'exemple de ce [chapitre](ch=fonctionsavancees).
On veut représenter un étudiant par un enregistrement à 4 champs `numero_etud`, `nom`, `prenom`, `diplome`.
Aucun type composé natif de python ne correspond à un enregistrement.

En pratique, on utilise alors un `dict` :

```python
etu = {"numero": 20231234, "nom": "Dupont", "prenom": "Alice", "notes": []}
```

**Limites de cette approche :**

- rien n'empêche d'oublier un champ, d'en ajouter un par erreur, ou de faire une faute de frappe sur une clé (`etu["nomm"]`) : l'erreur n'est détectée qu'à l'exécution, loin de sa cause ;
- les *traitements* associés à un étudiant (ajouter une note, calculer une moyenne) doivent être écrits comme des **fonctions séparées**, prenant le dictionnaire en paramètre : rien ne dit, en lisant le code, que ces fonctions et ce dictionnaire « vont ensemble » ;
- si on veut représenter plusieurs *sortes* d'étudiants (un étudiant boursier, un étudiant en alternance ...) qui partagent l'essentiel de leur comportement mais en diffèrent sur quelques points, un dictionnaire ne propose aucun mécanisme de réutilisation.

**Vocabulaire.** 

La **programmation orientée objet** (POO) est un paradigme de programmation qui répond à ces limites en regroupant, au sein d'une même entité appelée **classe** :

- des **données** : les **attributs** (comme les champs d'un enregistrement) ;
- des **traitements** : les **méthodes**, qui sont des fonctions *attachées* à la classe et qui agissent naturellement sur ses attributs.

Une **instance** (ou **objet**) est une valeur concrète construite à partir d'une classe -- de même qu'une valeur `3` est une instance du type `int`.

En python **tout est objet**, y compris les entiers, les chaînes de caractères et les listes déjà rencontrés !

```python
print(type(3), type("bonjour"), type([1, 2, 3]))
```

produit :

```
<class 'int'> <class 'str'> <class 'list'>
```

Les types `int`, `str`, `list`, … sont donc eux-mêmes des **classes**, prédéfinies par python. Ce chapitre montre comment définir **ses propres classes**.

(sec:classe-objets)=
## Classes et objets

### Définir une classe

**Syntaxe minimale :**

```python
class NomDeClasse:
    def __init__(self, param1, param2, ...):
        self.attribut1 = param1
        self.attribut2 = param2
        ...
```

- Le nom d'une classe s'écrit conventionnellement avec une **majuscule initiale** (`Etudiant`, et non `etudiant`).
- `__init__` est une méthode spéciale appelée le **constructeur** : elle est exécutée automatiquement lors de la création (*instanciation*) d'un objet, et sert à initialiser ses attributs.
- `self` désigne **l'instance elle-même** ; c'est **toujours le premier paramètre** de chaque méthode d'instance, et python le fournit automatiquement — on ne l'indique jamais explicitement lors de l'appel.
- `self.attribut = valeur` crée (ou modifie) l'attribut `attribut` de l'instance `self`, exactement comme on écrirait `etudiant.numero_etud` pour un enregistrement.

**Instancier** un objet, c'est appeler la classe comme une fonction :
> `objet = NomDeClasse(arg1, arg2, ...)`

Ceci déclenche l'appel à `__init__(self, arg1, arg2, ...)` où `self` est automatiquement l'objet en cours de création.

In [3]:
class Etudiant:
    """Représente un étudiant identifié par son numéro, son nom et son prénom."""

    def __init__(self, numero: int, nom: str, prenom: str):
        self.numero = numero
        self.nom = nom
        self.prenom = prenom
        self.notes = []  # liste des notes obtenues

    def ajouter_note(self, note: float):
        self.notes.append(note)

    def moyenne(self) -> float:
        if not self.notes:
            return 0.0
        return sum(self.notes) / len(self.notes)


etu1 = Etudiant(20231234, "Dupont", "Alice")
etu2 = Etudiant(20235678, "Martin", "Bob")

print(etu1.nom, etu1.prenom, etu1.numero)
print(type(etu1))

etu1.ajouter_note(14.5)
etu1.ajouter_note(16.0)
print(etu1.notes, etu1.moyenne())

print(etu1 is etu2, etu1 == etu2)

Dupont Alice 20231234
<class '__main__.Etudiant'>
[14.5, 16.0] 15.25
False False


**Rmq.**

- `etu1.nom` se lit et s'écrit exactement comme un champ d'enregistrement : `objet.attribut`.
- Chaque instance possède **son propre espace** pour ses attributs : modifier `etu1.notes` ne modifie pas `etu2.notes`.
- `etu1 == etu2` renvoie `False` : par défaut, `==` compare l'**identité** des objets (comme `is`), pas le contenu de leurs attributs. On verra en [section](sec:dunder) comment redéfinir ce comportement.
- `type(etu1)` confirme que `etu1` est bien une instance de la classe `Etudiant`, de la même façon que `type(3)` renvoie `int`.

**Attention.** `self` n'est pas un mot-clé du langage (contrairement à `def` ou `class`) : c'est une **convention** universellement respectée. Oublier de préfixer un attribut par `self.` est une erreur fréquente en début d'apprentissage :

In [4]:
class Compteur:
    def incrementer(self):
        self.valeur = self.valeur + 1   # `valeur` n'a jamais été initialisé !

c = Compteur()
try:
    c.incrementer()
except AttributeError as e:
    print("Erreur :", e)

Erreur : 'Compteur' object has no attribute 'valeur'


**Exercice.** Pourquoi cette erreur se produit-elle, alors que `self.valeur = self.valeur + 1` *écrit* aussi dans `self.valeur` ?

*Réponse :* en python, une expression `a = a + 1` évalue d'abord le membre de **droite** (`self.valeur + 1`), ce qui nécessite de *lire* `self.valeur` — qui n'existe pas encore, faute d'avoir été créé dans `__init__`. Il faut initialiser `self.valeur` (par exemple à `0`) dans le constructeur de la classe.

### Activité 1 — Une classe `Point`

**Énoncé.** Écrire une classe `Point` qui modélise un point du plan, avec :

- un constructeur `__init__(self, x: float = 0.0, y: float = 0.0)` qui initialise les attributs `x` et `y` ;
- une méthode `distance(self, autre: "Point") -> float` qui renvoie la distance euclidienne entre `self` et `autre` (utiliser le module `math`) ;
- une méthode `translater(self, dx: float, dy: float) -> None` qui déplace le point de `dx` selon `x` et `dy` selon `y` (modification *en place*, comme une méthode `.append()` sur une liste).

Tester la classe avec deux points `p1 = Point(0, 0)` et `p2 = Point(3, 4)`.

**À vous de jouer avant de lire la correction ci-dessous !**

In [5]:
import math

class Point:
    def __init__(self, x: float = 0.0, y: float = 0.0):
        self.x = x
        self.y = y

    def distance(self, autre: "Point") -> float:
        return math.sqrt((self.x - autre.x) ** 2 + (self.y - autre.y) ** 2)

    def translater(self, dx: float, dy: float) -> None:
        self.x += dx
        self.y += dy


p1 = Point(0, 0)
p2 = Point(3, 4)
print(p1.distance(p2))

p1.translater(1, 1)
print(p1.x, p1.y)

5.0
1 1


**Rmq.** L'annotation de type `autre: "Point"` est écrite entre guillemets (une *chaîne de caractères*) car, au moment où python lit la définition de la méthode `distance`, la classe `Point` n'est pas encore entièrement définie. C'est une pratique courante pour les méthodes qui référencent leur propre classe.

(sec:encapsulation)=
##  Encapsulation

**Vocabulaire.** L'**encapsulation** consiste à protéger l'accès direct aux attributs d'un objet, pour n'autoriser leur lecture ou leur modification qu'au travers de méthodes qui en contrôlent la cohérence.

Contrairement à d'autres langages (Java, C++), python **n'impose aucune restriction stricte** d'accès aux attributs : c'est une affaire de **convention**, respectée par les autres développeurs et développeuses.

| Convention          | Signification                                                                 |
| -------------------- | ------------------------------------------------------------------------------ |
| `attribut`            | attribut **public** : utilisable librement depuis l'extérieur de la classe    |
| `_attribut`           | attribut **protégé** *par convention* : « usage interne », mais accessible     |
| `__attribut`          | attribut **« privé »** : python renomme l'attribut (*name mangling*), rendant son accès direct malaisé depuis l'extérieur |

### Property : contrôler l'accès à un attribut

Le décorateur `@property` permet de définir une méthode qui **se comporte comme un attribut** en lecture (pas de parenthèses à l'appel), et `@nom.setter` fait de même pour l'écriture, en y ajoutant un contrôle.

In [8]:
class CompteBancaire:
    def __init__(self, titulaire: str, solde: float = 0.0):
        self.titulaire = titulaire
        self._solde = solde   # attribut "protégé" par convention

    @property
    def solde(self) -> float:
        return self._solde

    @solde.setter
    def solde(self, valeur: float) -> None:
        if valeur < 0:
            raise ValueError("le solde ne peut pas être négatif")
        self._solde = valeur

    def deposer(self, montant: float) -> None:
        self.solde = self.solde + montant

    def retirer(self, montant: float) -> None:
        if montant > self._solde:
            raise ValueError("solde insuffisant")
        self.solde = self.solde - montant


c = CompteBancaire("Alice", 100)
c.deposer(50)
print(c.solde)

c.retirer(30)
print(c.solde)

try:
    c.solde = -10
except ValueError as e:
    print("Erreur :", e)

try:
    c.retirer(10000)
except ValueError as e:
    print("Erreur :", e)

150
120
Erreur : le solde ne peut pas être négatif
Erreur : solde insuffisant


**Rmq.**

- `c.solde` se lit **sans parenthèses**, comme un attribut ordinaire, bien qu'il s'agisse en réalité de l'appel d'une méthode.
- `c.solde = -10` déclenche le *setter*, qui refuse la valeur et lève une exception : le solde reste cohérent.
- Cette approche permet de **changer d'avis plus tard** (ajouter un contrôle) sans casser le code qui utilise déjà `c.solde` — un avantage important par rapport à l'accès direct `c._solde`.

### ($\star\$) *Name mangling* des attributs `__attribut`

Un attribut préfixé par **deux** *underscores* (et se terminant par au plus un underscore) est automatiquement renommé par python en `_NomDeClasse__attribut`, ce qui rend son accès direct depuis l'extérieur peu pratique — mais **toujours possible**, la « vie privée » n'étant jamais totale en python.

In [9]:
class Secret:
    def __init__(self):
        self.__valeur = 42

    def afficher(self):
        print(self.__valeur)

s = Secret()
s.afficher()
print(s.__dict__)

try:
    print(s.__valeur)
except AttributeError as e:
    print("Erreur :", e)

print(s._Secret__valeur)   # toujours accessible, mais fortement déconseillé

42
{'_Secret__valeur': 42}
Erreur : 'Secret' object has no attribute '__valeur'
42


**Rmq.** En pratique, la convention `_attribut` (un seul underscore) suffit dans la grande majorité des cas pour signaler « ceci est un détail d'implémentation, ne pas y toucher directement » ; on réserve `__attribut` aux cas où l'on veut vraiment éviter les collisions de noms, en particulier avec l'héritage -- voir cette [section](sec:heritage).

(sec:dunder)=
##  Méthodes spéciales (*dunder*)


**Vocabulaire.** Les méthodes dont le nom est encadré de deux *underscores* (`__init__`, `__str__`, …) sont appelées **méthodes spéciales** ou *dunder methods* (de l'anglais *double underscore*). Elles ne sont (presque) jamais appelées explicitement : python les appelle **implicitement** pour donner du sens à des opérateurs ou fonctions natives (`print()`, `==`, `<`, `+`, `len()`, …) appliqués à vos objets.

| Méthode spéciale | Déclenchée par             | Rôle                                             |
| ----------------- | --------------------------- | -------------------------------------------------- |
| `__str__(self)`   | `print(obj)`, `str(obj)`   | représentation **lisible**, destinée à l'utilisateur |
| `__repr__(self)`  | `repr(obj)`, invite interactive, affichage d'une `list` d'objets | représentation **non ambigüe**, destinée au développeur |
| `__eq__(self, autre)` | `obj1 == obj2`          | égalité *par valeur* plutôt que par identité       |
| `__lt__(self, autre)` | `obj1 < obj2`, `sorted()` | ordre *strict* entre deux objets                  |
| `__add__(self, autre)` | `obj1 + obj2`           | addition (à définir selon le sens voulu)           |
| `__len__(self)`   | `len(obj)`                  | « taille » de l'objet                              |

C'est en redéfinissant ces méthodes qu'une classe devient **« pythonique »** : elle s'intègre naturellement aux fonctions et opérateurs déjà connus, plutôt que d'exiger des fonctions dédiées.

In [ ]:
class Point:
    def __init__(self, x=0.0, y=0.0):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

    def __eq__(self, autre):
        return isinstance(autre, Point) and self.x == autre.x and self.y == autre.y

    def __lt__(self, autre):
        return (self.x ** 2 + self.y ** 2) < (autre.x ** 2 + autre.y ** 2)

    def __add__(self, autre):
        return Point(self.x + autre.x, self.y + autre.y)

    def __len__(self):
        return 2


p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

print(p1)               # utilise __str__
print(repr(p1))          # utilise __repr__
print(p1 == p2, p1 == p3)  # utilise __eq__
print(p1 < p3)            # utilise __lt__
print(p1 + p3)             # utilise __add__, puis __str__ pour l'affichage
print(len(p1))              # utilise __len__

liste_points = [p3, p1, Point(0, 0)]
print(sorted(liste_points))  # utilise __lt__ ; l'affichage de la liste utilise __repr__

(1, 2)
Point(1, 2)
True False
True
(4, 6)
2
[Point(0, 0), Point(1, 2), Point(3, 4)]


**Rmq.**

- Sans `__eq__`, `p1 == p2` aurait renvoyé `False` (comparaison d'identité), comme observé pour `Etudiant` en section [Classe et objets](sec:classe-objets).
- `print()` d'une liste d'objets appelle `__repr__` sur chacun d'eux, et non `__str__` : c'est pourquoi il est recommandé de **toujours définir `__repr__`**, `__str__` étant optionnelle (python utilise `__repr__` par défaut si `__str__` est absente).
- Si l'on définit `__eq__`, il est également recommandé de définir `__hash__` pour continuer à pouvoir utiliser l'objet comme clé de `dict` ou élément de `set` (voir la documentation officielle) — ce point n'est pas approfondi ici.

**Exercice.** Quelle méthode spéciale faudrait-il définir pour permettre l'écriture `p1 in liste_points`, sachant que cette syntaxe existe déjà pour les listes vues en L1 ?

*Réponse :* `__contains__(self, valeur)`. À défaut, python retombe sur une itération utilisant `__eq__`, ce qui fonctionne également ici grâce à la définition de `__eq__` ci-dessus.

(sec:attributs-classe)=
##  Attributs et méthodes de classe

Jusqu'ici, tous les attributs étaient des **attributs d'instance** : chaque objet possède sa propre valeur. Un **attribut de classe**, déclaré directement dans le corps de la classe (hors de toute méthode), est en revanche **partagé par toutes les instances**.

- `@classmethod` définit une méthode dont le premier paramètre, nommé par convention `cls`, désigne la **classe elle-même** (et non une instance) ; utile par exemple pour des constructeurs alternatifs ou pour manipuler un attribut de classe.
- `@staticmethod` définit une méthode qui n'utilise ni `self` ni `cls` : elle est rattachée à la classe par pure commodité d'organisation du code (regrouper une fonction utilitaire avec la classe à laquelle elle se rapporte).

In [1]:
class Etudiant:
    nb_etudiants = 0  # attribut de CLASSE : une seule valeur, partagée

    def __init__(self, nom: str):
        self.nom = nom            # attribut D'INSTANCE
        Etudiant.nb_etudiants += 1

    @classmethod
    def nombre_etudiants(cls) -> int:
        return cls.nb_etudiants

    @staticmethod
    def est_nom_valide(nom: str) -> bool:
        return isinstance(nom, str) and nom.isalpha()


e1 = Etudiant("Alice")
e2 = Etudiant("Bob")

print(Etudiant.nb_etudiants, e1.nb_etudiants)   # accessible via la classe OU une instance
print(Etudiant.nombre_etudiants())
print(Etudiant.est_nom_valide("Alice"), Etudiant.est_nom_valide("A1ice"))

2 2
2
True False


**Attention.** `Etudiant.nb_etudiants += 1` (et non `self.nb_etudiants += 1`) est essentiel : cette dernière écriture créerait un **nouvel attribut d'instance** `nb_etudiants`, propre à `self`, masquant l'attribut de classe sans le modifier — un piège classique.

**Rmq.** On retrouve ici, en écho au chapitre 10 de L1 (« Affectation et appel de fonction : aspects avancés »), une distinction analogue à celle entre variable locale et variable globale d'une fonction.

(sec:heritage)=
##  Héritage

**Vocabulaire.** L'**héritage** permet de définir une nouvelle classe (dite **classe fille** ou **sous-classe**) à partir d'une classe existante (dite **classe mère** ou **classe de base**), en **réutilisant** ses attributs et méthodes, et en pouvant en **ajouter** de nouveaux ou en **redéfinir** certains (on parle de **redéfinition** ou *override*).

**Syntaxe :**

```python
class ClasseFille(ClasseMere):
    def __init__(self, ..., attribut_specifique):
        super().__init__(...)              # appelle le constructeur de la classe mère
        self.attribut_specifique = attribut_specifique
```

`super()` désigne la classe mère et permet d'appeler ses méthodes (typiquement `__init__`) sans avoir à répéter leur code.

In [ ]:
class Animal:
    def __init__(self, nom: str):
        self.nom = nom

    def se_deplacer(self) -> str:
        return f"{self.nom} se déplace."

    def crier(self) -> str:
        return f"{self.nom} fait un bruit."

    def __str__(self):
        return f"{type(self).__name__}({self.nom})"


class Chien(Animal):        # Chien HERITE de Animal
    def crier(self) -> str:    # redéfinition (override) de la méthode crier
        return f"{self.nom} aboie !"


class Chat(Animal):
    def __init__(self, nom: str, sterilise: bool = False):
        super().__init__(nom)       # réutilise le constructeur de Animal
        self.sterilise = sterilise  # attribut supplémentaire, propre à Chat

    def crier(self) -> str:
        return f"{self.nom} miaule !"


rex = Chien("Rex")
felix = Chat("Félix", sterilise=True)

for a in (rex, felix, Animal("Bêta")):
    print(a, "->", a.crier(), a.se_deplacer())

print(isinstance(rex, Animal), isinstance(felix, Chien))
print(issubclass(Chat, Animal))

Chien(Rex) -> Rex aboie ! Rex se déplace.
Chat(Félix) -> Félix miaule ! Félix se déplace.
Animal(Bêta) -> Bêta fait un bruit. Bêta se déplace.
True False
True


**Rmq.**

- `Chien` ne redéfinit pas `__init__` : elle **hérite tel quel** du constructeur de `Animal`.
- `Chien` ne redéfinit pas `se_deplacer` ni `__str__` non plus : ces méthodes sont **héritées** et fonctionnent sans modification.
- `type(self).__name__` (dans `Animal.__str__`) renvoie le nom de la classe **réelle** de l'instance (`"Chien"`, `"Chat"` ou `"Animal"`), même si le code de `__str__` est écrit une seule fois, dans `Animal`.
- `isinstance(felix, Chien)` renvoie `False` : `felix` est un `Chat`, pas un `Chien`, même si les deux héritent de `Animal`.
- `issubclass(Chat, Animal)` teste une relation entre **classes** (et non entre objets et classes, comme `isinstance`).

(sec:polymorphisme)=
##  Polymorphisme

**Vocabulaire.** Le **polymorphisme** (littéralement « plusieurs formes ») désigne la capacité d'un même code à traiter de façon uniforme des objets de classes différentes, dès lors qu'ils répondent aux mêmes méthodes.

En python, le polymorphisme est **naturel** et ne nécessite aucune déclaration particulière : c'est ce qu'on appelle le ***duck typing*** (« si ça marche comme un canard et que ça cancane comme un canard, alors s'en est un ») — seul compte le fait qu'un objet **possède** la méthode appelée, indépendamment de sa classe exacte.

In [ ]:
def faire_du_bruit(animaux) -> None:
    for a in animaux:
        print(a.crier())   # fonctionne quelle que soit la classe REELLE de `a`

faire_du_bruit([rex, felix])

Rex aboie !
Félix miaule !


**Rmq.** La fonction `faire_du_bruit` n'a **jamais besoin de savoir** si `a` est un `Chien`, un `Chat` ou une autre sous-classe de `Animal` : elle appelle simplement `a.crier()`, et c'est **la classe réelle de l'objet** qui détermine, à l'exécution, quelle version de la méthode est exécutée. C'est exactement le même principe que la *surcharge* de fonctions rencontrée en L1 (chapitre 7) avec le typage des paramètres, mais réalisé ici *dynamiquement*, au travers de l'héritage.

### Activité 2 — Une hiérarchie de véhicules

**Énoncé.** En s'inspirant de la hiérarchie `Animal` / `Chien` / `Chat` :

1. Définir une classe mère `Vehicule` avec un constructeur `__init__(self, marque: str, vitesse_max: float)` et une méthode `decrire(self) -> str` qui renvoie une phrase du type `"Vehicule Peugeot (vitesse max 180 km/h)"` (utiliser `type(self).__name__`).
2. Définir deux classes filles `Voiture` (attribut supplémentaire `nb_portes`) et `Moto` (attribut supplémentaire `cylindree`), qui **redéfinissent** `decrire` pour ajouter leur information spécifique **tout en réutilisant** `decrire` de `Vehicule` grâce à `super()`.
3. Construire une liste `parc` mélangeant des `Voiture`, des `Moto` et un `Vehicule` « générique », et afficher la description de chaque véhicule avec **une seule boucle** — illustrant le polymorphisme.

**À vous de jouer avant de lire la correction indicative ci-dessous !**

In [ ]:
class Vehicule:
    def __init__(self, marque: str, vitesse_max: float):
        self.marque = marque
        self.vitesse_max = vitesse_max

    def decrire(self) -> str:
        return f"{type(self).__name__} {self.marque} (vitesse max {self.vitesse_max} km/h)"


class Voiture(Vehicule):
    def __init__(self, marque: str, vitesse_max: float, nb_portes: int):
        super().__init__(marque, vitesse_max)
        self.nb_portes = nb_portes

    def decrire(self) -> str:
        base = super().decrire()               # réutilise la description de Vehicule
        return base + f", {self.nb_portes} portes"


class Moto(Vehicule):
    def __init__(self, marque: str, vitesse_max: float, cylindree: int):
        super().__init__(marque, vitesse_max)
        self.cylindree = cylindree

    def decrire(self) -> str:
        base = super().decrire()
        return base + f", {self.cylindree} cm3"


parc = [Voiture("Peugeot", 180, 5), Moto("Yamaha", 220, 600), Vehicule("Générique", 100)]
for v in parc:
    print(v.decrire())   # une seule boucle, une seule ligne : polymorphisme

Voiture Peugeot (vitesse max 180 km/h), 5 portes
Moto Yamaha (vitesse max 220 km/h), 600 cm3
Vehicule Générique (vitesse max 100 km/h)


**Rmq.** L'appel `super().decrire()` **dans** `Voiture.decrire` illustre une bonne pratique : on **complète** le comportement de la classe mère plutôt que de **dupliquer** son code — un principe déjà rencontré en L1 sous la forme générale de la *décomposition en fonctions* (chapitres 1 et 2), appliqué ici entre une méthode et sa redéfinition.

(sec:abc)=
##  (\(\star\)) Classes abstraites

Une **classe abstraite** définit une **interface commune** — un ensemble de méthodes que toute sous-classe **doit** implémenter — sans fournir elle-même d'implémentation complète. Elle ne peut **pas** être instanciée directement.

En python, le module `abc` (*Abstract Base Classes*) fournit la classe `ABC` et le décorateur `@abstractmethod`.

In [ ]:
import math
from abc import ABC, abstractmethod

class Forme(ABC):
    @abstractmethod
    def aire(self) -> float:
        ...

    @abstractmethod
    def perimetre(self) -> float:
        ...

    def __str__(self):
        return f"{type(self).__name__}(aire={self.aire():.2f}, périmètre={self.perimetre():.2f})"


class Rectangle(Forme):
    def __init__(self, largeur: float, hauteur: float):
        self.largeur = largeur
        self.hauteur = hauteur

    def aire(self) -> float:
        return self.largeur * self.hauteur

    def perimetre(self) -> float:
        return 2 * (self.largeur + self.hauteur)


class Cercle(Forme):
    def __init__(self, rayon: float):
        self.rayon = rayon

    def aire(self) -> float:
        return math.pi * self.rayon ** 2

    def perimetre(self) -> float:
        return 2 * math.pi * self.rayon


formes = [Rectangle(3, 4), Cercle(2)]
for f in formes:
    print(f)    # utilise __str__, qui utilise lui-même aire() et perimetre() redéfinies

try:
    f = Forme()
except TypeError as e:
    print("Erreur :", e)

Rectangle(aire=12.00, périmètre=14.00)
Cercle(aire=12.57, périmètre=12.57)
Erreur : Can't instantiate abstract class Forme without an implementation for abstract methods 'aire', 'perimetre'


**Rmq.**

- `Forme()` est **impossible** : python refuse d'instancier une classe abstraite tant que toutes ses méthodes `@abstractmethod` n'ont pas été redéfinies.
- En revanche, `Forme.__str__` — qui n'est **pas** abstraite — est bien héritée et utilisable par `Rectangle` et `Cercle`, alors même qu'elle appelle `self.aire()` et `self.perimetre()`, deux méthodes qui n'existent, *au moment de l'écriture de `Forme`*, que sous forme de « promesses » tenues plus tard par les sous-classes. C'est un exemple supplémentaire de polymorphisme.
- Une classe abstraite est utile pour **garantir**, dès la conception, qu'une famille de classes partage une interface commune — utile en particulier sur de gros projets à plusieurs développeurs et développeuses.

## . Vérification rapide de la compréhension

Répondez aux questions suivantes **avant** de consulter les réponses.

**Q1.** Que renvoie `p1 == p2` pour deux objets `p1` et `p2` d'une classe qui **ne redéfinit pas** `__eq__` ?

a. `True` si tous les attributs de `p1` et `p2` ont la même valeur
b. `True` uniquement si `p1` et `p2` sont le **même** objet en mémoire
c. Une erreur, car `==` n'est pas défini par défaut

*Réponse : **b**. Sans redéfinition de `__eq__`, `==` compare l'identité des objets, comme `is` (voir [section](sec:classe-objets) avec `etu1 == etu2`).*

**Q2.** Dans une méthode d'instance, à quoi correspond le paramètre `self` ?

a. À la classe elle-même
b. À l'instance sur laquelle la méthode est appelée
c. À un attribut particulier de la classe

*Réponse : **b**. C'est pourquoi `@classmethod` utilise `cls` (la classe) plutôt que `self` (une instance) — voir [section](sec:attributs-classe).*

**Q3.** Pourquoi `Etudiant.nb_etudiants += 1` et non `self.nb_etudiants += 1` dans le constructeur, pour incrémenter un compteur **partagé** par toutes les instances ?

*Réponse : `self.nb_etudiants += 1` créerait un nouvel attribut D'INSTANCE `nb_etudiants` propre à `self`, masquant l'attribut de classe sans le modifier réellement — voir la remarque de la voir [section](sec:attributs-classe).*

**Q4.** Une classe `Carre` hérite de `Rectangle`. Que fait `super().__init__(cote, cote)` dans le constructeur de `Carre` ?

a. Elle crée une nouvelle instance de `Rectangle`
b. Elle appelle le constructeur de `Rectangle` pour initialiser les attributs hérités
c. Elle redéfinit la méthode `__init__` de `Rectangle`

*Réponse : **b**. `super()` donne accès aux méthodes de la classe mère, notamment pour réutiliser (et non dupliquer) son constructeur — voir sections [héritage](sec:heritage) et  [polymorphisme](sec:polymorphisme).*

**Q5.** Pourquoi ne peut-on pas écrire `f = Forme()` si `Forme` hérite de `ABC` et définit des méthodes `@abstractmethod` ?

*Réponse : une classe abstraite ne peut pas être instanciée directement tant que toutes ses méthodes abstraites n'ont pas été redéfinies par une sous-classe concrète — voir section [abc](sec:abc).*

**Q6.** Quelle est la différence entre `__str__` et `__repr__` ?

*Réponse : `__str__` fournit une représentation lisible destinée à l'utilisateur (via `print()`), tandis que `__repr__` fournit une représentation non ambigüe destinée au développeur, utilisée notamment par défaut pour afficher les éléments d'une `list` — voir section [méthodes spéciales](sec:dunder)*

## Synthèse et compétences

### Avoir les idées claires

- Une **classe** regroupe des **attributs** (données) et des **méthodes** (traitements), et généralise la notion d'enregistrement vue en L1.
- Une **instance** (ou objet) est une valeur concrète construite à partir d'une classe, via son constructeur `__init__`.
- `self` désigne toujours l'instance courante et doit être le premier paramètre de chaque méthode d'instance.
- L'**encapsulation** protège la cohérence des attributs (`_attribut`, `__attribut`, `@property`).
- Les **méthodes spéciales** (`__str__`, `__eq__`, `__lt__`, …) intègrent une classe aux opérateurs et fonctions natives de python.
- Les **attributs de classe** sont partagés par toutes les instances ; les **attributs d'instance** sont propres à chacune.
- L'**héritage** (`class Fille(Mere):`, `super()`) permet de réutiliser et d'étendre une classe existante.
- Le **polymorphisme** permet à un même code de traiter uniformément des objets de classes différentes, dès lors qu'ils répondent aux mêmes méthodes.
- (\(\star\)) Une **classe abstraite** (`abc.ABC`, `@abstractmethod`) impose une interface commune à une famille de classes.

### Savoir faire

- Définir une classe avec `__init__` et des méthodes, et instancier des objets.
- Choisir entre attribut public, protégé (`_`) ou « privé » (`__`), et définir une `@property` lorsqu'un contrôle est nécessaire.
- Redéfinir les méthodes spéciales usuelles pour rendre une classe pythonique.
- Construire une hiérarchie de classes avec héritage, en utilisant `super()` pour réutiliser le comportement de la classe mère.
- Écrire une fonction qui exploite le polymorphisme, sans tester explicitement le type de ses arguments.
- ($\star$) Définir une classe abstraite pour garantir une interface commune.